# 06 · Búsqueda binaria, ordenamiento, golosos, pilas, colas, conjuntos y mapas

Tópico 06 del temario. Las herramientas más poderosas de este nivel.

## Ordenamiento

`sorted(a)` devuelve una copia ordenada; `a.sort()` ordena en el sitio. Ambos son `O(n log n)` y **estables** (los empates mantienen su orden original).

In [ ]:
a = [5, 2, 8, 1, 9]

assert sorted(a) == [1, 2, 5, 8, 9]
assert sorted(a, reverse=True) == [9, 8, 5, 2, 1]

# key: por qué criterio ordenar
palabras = ["pera", "uva", "manzana", "kiwi"]
assert sorted(palabras, key=len) == ["uva", "pera", "kiwi", "manzana"]       # por longitud (estable)
assert sorted(palabras, key=lambda p: (len(p), p)) == ["uva", "kiwi", "pera", "manzana"]

# Ordenar pares por segundo elemento, luego por el primero
pares = [(1, 5), (2, 3), (3, 5), (4, 1)]
assert sorted(pares, key=lambda p: (p[1], p[0])) == [(4, 1), (2, 3), (1, 5), (3, 5)]
assert sorted(pares, key=lambda p: -p[1]) == [(1, 5), (3, 5), (2, 3), (4, 1)]       # descendente por [1]

# Ordenar índices por valor
valores = [30, 10, 20]
orden = sorted(range(len(valores)), key=lambda i: valores[i])
assert orden == [1, 2, 0]

### Ordenamiento por conteo

Cuando los valores son pequeños (por ejemplo, `0 ≤ x ≤ 10^6`), contar es más rápido que comparar: `O(n + valor_max)`.

In [ ]:
def orden_por_conteo(arr, maximo):
    cuenta = [0] * (maximo + 1)
    for x in arr:
        cuenta[x] += 1
    res = []
    for v, c in enumerate(cuenta):
        res.extend([v] * c)
    return res


assert orden_por_conteo([3, 1, 4, 1, 5, 9, 2, 6], 9) == [1, 1, 2, 3, 4, 5, 6, 9]

### ¿Qué cambia cuando está ordenado?

Ordenar suele **simplificar** el problema: los iguales quedan juntos, el mínimo y el máximo están en los extremos, se puede usar búsqueda binaria y dos punteros.

In [ ]:
def hay_repetidos(arr):
    a = sorted(arr)
    return any(a[i] == a[i + 1] for i in range(len(a) - 1))


def mediana(arr):
    a = sorted(arr)
    n = len(a)
    return a[n // 2] if n % 2 else (a[n // 2 - 1] + a[n // 2]) / 2


assert hay_repetidos([3, 1, 3]) and not hay_repetidos([3, 1, 2])
assert mediana([5, 1, 3]) == 3 and mediana([4, 1, 3, 2]) == 2.5

## Búsqueda binaria

En un arreglo **ordenado**, mira el elemento del medio: eso te dice si lo que buscas está a la izquierda o a la derecha. Descartas la mitad cada vez: `O(log n)`.

> Con 10^9 elementos solo hacen falta unos **30** pasos.

In [ ]:
def busqueda_binaria(arr, x):
    izq, der = 0, len(arr) - 1
    while izq <= der:
        medio = (izq + der) // 2
        if arr[medio] == x:
            return medio
        if arr[medio] < x:
            izq = medio + 1
        else:
            der = medio - 1
    return -1


a = [1, 3, 5, 7, 9, 11]
assert busqueda_binaria(a, 7) == 3
assert busqueda_binaria(a, 4) == -1
assert busqueda_binaria([], 4) == -1

### Primera posición donde se cumple algo (la forma más útil)

Plantilla: la condición es `False False False True True True`. Encuentra el **primer** `True`.

In [ ]:
def primer_mayor_o_igual(arr, x):
    """Primer índice i con arr[i] >= x (len(arr) si no existe)."""
    izq, der = 0, len(arr)
    while izq < der:
        medio = (izq + der) // 2
        if arr[medio] >= x:
            der = medio
        else:
            izq = medio + 1
    return izq


a = [1, 3, 3, 5, 7]
assert primer_mayor_o_igual(a, 3) == 1
assert primer_mayor_o_igual(a, 4) == 3
assert primer_mayor_o_igual(a, 100) == 5

# La librería estándar lo trae hecho
from bisect import bisect_left, bisect_right

assert bisect_left(a, 3) == 1            # primer índice con arr[i] >= 3
assert bisect_right(a, 3) == 3           # primer índice con arr[i] > 3
assert bisect_right(a, 3) - bisect_left(a, 3) == 2       # cuántas veces aparece el 3

### Búsqueda binaria sobre la respuesta

Si preguntar "¿**funciona** el valor `x`?" es fácil, y la respuesta pasa de *no* a *sí* al crecer `x`, haz búsqueda binaria sobre `x`.

Ejemplo: `n` cuerdas de distintos largos; cortarlas en trozos iguales de largo `L`. ¿Cuál es el mayor `L` entero que da al menos `k` trozos?

In [ ]:
def mayor_trozo(cuerdas, k):
    def alcanza(L):
        return sum(c // L for c in cuerdas) >= k

    izq, der = 0, max(cuerdas)          # izq siempre "funciona" (0 = nada), buscamos el mayor que funcione
    while izq < der:
        medio = (izq + der + 1) // 2    # +1 para no quedar en bucle infinito
        if medio > 0 and alcanza(medio):
            izq = medio
        else:
            der = medio - 1
    return izq


assert mayor_trozo([9, 7, 5], 5) == 3         # 3+2+1 = 6 trozos con L=3; con L=4 serían 4
assert mayor_trozo([4], 5) == 0

## Algoritmos golosos (greedy)

**Idea:** en cada paso toma la decisión que **parece mejor ahora**, sin volver atrás. No siempre funciona; cuando sí, es muy rápido. Casi siempre empieza con **ordenar**.

### Monedas (cuando cada moneda es múltiplo de la anterior)

In [ ]:
def minimo_monedas(monto, monedas):
    cuenta = 0
    for m in sorted(monedas, reverse=True):
        cuenta += monto // m
        monto %= m
    return cuenta if monto == 0 else -1


assert minimo_monedas(87, [1, 5, 10, 25, 50]) == 5          # 50+25+10+1+1
assert minimo_monedas(7, [5, 2]) == 2

# ⚠️ Con monedas "raras" el goloso falla: monto 6 con [1, 3, 4] -> goloso da 4+1+1 (3), pero 3+3 son 2

### Selección de actividades

Varias actividades con hora de inicio y fin. ¿Cuántas puedes hacer como máximo sin que se traslapen?

**Estrategia correcta:** ordenar por hora de **fin** y quedarte con la siguiente que empiece después de la última elegida.

In [ ]:
def max_actividades(intervalos):
    contador = 0
    fin_ultima = float("-inf")
    for inicio, fin in sorted(intervalos, key=lambda x: x[1]):
        if inicio >= fin_ultima:
            contador += 1
            fin_ultima = fin
    return contador


assert max_actividades([(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11)]) == 3      # (1,4) (5,7) (8,11)
assert max_actividades([]) == 0

### Otros golosos clásicos

In [ ]:
# Mayor número formado con las cifras de la lista: ordenar como texto, de mayor a menor
def mayor_numero(cifras):
    return int("".join(sorted(map(str, cifras), reverse=True)))


assert mayor_numero([3, 9, 1, 7]) == 9731


# Emparejar para minimizar la suma de diferencias: ordenar ambos y emparejar uno a uno
def minima_diferencia_total(a, b):
    return sum(abs(x - y) for x, y in zip(sorted(a), sorted(b)))


assert minima_diferencia_total([4, 1, 8], [2, 9, 3]) == 1 + 1 + 1


# Terminar tareas con plazo: hacer primero las cortas minimiza el tiempo de espera total
def espera_total(duraciones):
    tiempo = 0
    total = 0
    for d in sorted(duraciones):
        total += tiempo
        tiempo += d
    return total


assert espera_total([3, 1, 2]) == 0 + 1 + 3

## Pilas (stack): el último en entrar es el primero en salir

En Python, una **lista** con `append` y `pop`, ambos `O(1)`. Piensa en una pila de platos.

In [ ]:
pila = []
pila.append(1)
pila.append(2)
pila.append(3)
assert pila[-1] == 3             # mirar la cima
assert pila.pop() == 3 and pila == [1, 2]

### Paréntesis balanceados

La pila recuerda qué se abrió. Al ver un cierre, debe coincidir con el último apertura.

In [ ]:
def balanceado(s):
    pares = {")": "(", "]": "[", "}": "{"}
    pila = []
    for c in s:
        if c in "([{":
            pila.append(c)
        elif c in pares:
            if not pila or pila.pop() != pares[c]:
                return False
    return not pila


assert balanceado("([]{})") and balanceado("")
assert not balanceado("(]") and not balanceado("((") and not balanceado("())")

### Siguiente mayor elemento (pila monótona)

Para cada elemento, el primer elemento mayor a su derecha. Con la pila es `O(n)` en vez de `O(n²)`.

In [ ]:
def siguiente_mayor(arr):
    res = [-1] * len(arr)
    pila = []                          # índices cuyos "siguiente mayor" aún no encontramos
    for i, x in enumerate(arr):
        while pila and arr[pila[-1]] < x:
            res[pila.pop()] = x
        pila.append(i)
    return res


assert siguiente_mayor([2, 1, 5, 3, 4]) == [5, 5, -1, 4, -1]

### Evaluar una expresión en notación postfija

`3 4 + 2 *` significa `(3 + 4) * 2`.

In [ ]:
def evaluar_postfija(tokens):
    pila = []
    for t in tokens:
        if t in "+-*":
            b, a = pila.pop(), pila.pop()
            pila.append(a + b if t == "+" else a - b if t == "-" else a * b)
        else:
            pila.append(int(t))
    return pila[0]


assert evaluar_postfija("3 4 + 2 *".split()) == 14

## Colas (queue): el primero en entrar es el primero en salir

Usa `collections.deque`: `append` y `popleft` son `O(1)`. **No uses `list.pop(0)`** (es `O(n)`).

In [ ]:
from collections import deque

cola = deque()
cola.append(1)
cola.append(2)
cola.append(3)
assert cola.popleft() == 1 and list(cola) == [2, 3]

### Búsqueda en anchura (BFS): camino más corto en una cuadrícula

Con una cola exploras por "capas": primero a distancia 1, luego 2, etc. La primera vez que llegas a una celda es por el camino más corto.

In [ ]:
def camino_mas_corto(grid):
    """grid de '.' (libre) y '#' (pared). De (0,0) a la última celda. -1 si es imposible."""
    f, c = len(grid), len(grid[0])
    if grid[0][0] == "#" or grid[f - 1][c - 1] == "#":
        return -1
    dist = [[-1] * c for _ in range(f)]
    dist[0][0] = 0
    cola = deque([(0, 0)])
    while cola:
        i, j = cola.popleft()
        if (i, j) == (f - 1, c - 1):
            return dist[i][j]
        for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < f and 0 <= nj < c and grid[ni][nj] == "." and dist[ni][nj] == -1:
                dist[ni][nj] = dist[i][j] + 1
                cola.append((ni, nj))
    return -1


assert camino_mas_corto(["..#", "#..", "..."]) == 4
assert camino_mas_corto([".#", "#."]) == -1
assert camino_mas_corto(["."]) == 0

### Máximo de cada ventana (cola doble)

`deque` también permite `append`/`pop` por los dos lados. Mantiene los candidatos a máximo en orden decreciente.

In [ ]:
def maximos_ventana(arr, k):
    q = deque()                              # índices; arr[q[0]] es el máximo de la ventana
    res = []
    for i, x in enumerate(arr):
        while q and arr[q[-1]] <= x:
            q.pop()
        q.append(i)
        if q[0] <= i - k:                    # el primero salió de la ventana
            q.popleft()
        if i >= k - 1:
            res.append(arr[q[0]])
    return res


assert maximos_ventana([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]

## Conjuntos (`set`): sin repetidos y búsqueda `O(1)`

In [ ]:
s = {3, 1, 4, 1, 5}
assert s == {1, 3, 4, 5}
s.add(9)
s.discard(3)
assert 4 in s and 3 not in s

A, B = {1, 2, 3}, {2, 3, 4}
assert A & B == {2, 3}          # intersección
assert A | B == {1, 2, 3, 4}    # unión
assert A - B == {1}             # diferencia
assert A ^ B == {1, 4}          # en uno solo


def contar_distintos(arr):
    return len(set(arr))


def hay_repetidos_set(arr):
    return len(set(arr)) != len(arr)


assert contar_distintos([1, 2, 2, 3]) == 3 and hay_repetidos_set([1, 2, 2])

## Mapas (`dict`): clave → valor

In [ ]:
d = {}
d["ana"] = 3
d["luis"] = d.get("luis", 0) + 1       # contar sin error si no existe
assert d == {"ana": 3, "luis": 1}

from collections import Counter, defaultdict

assert Counter("abracadabra").most_common(1) == [("a", 5)]

grupos = defaultdict(list)             # valor por defecto: lista vacía
for palabra in ["roma", "amor", "gato", "toga", "mora"]:
    grupos["".join(sorted(palabra))].append(palabra)
assert sorted(grupos["amor"]) == ["amor", "mora", "roma"]

### Dos suma con mapa: `O(n)`

¿Existen dos números distintos (por posición) que sumen `k`? Para cada `x`, pregunta si ya viste `k - x`.

In [ ]:
def dos_suma(arr, k):
    visto = set()
    for x in arr:
        if k - x in visto:
            return True
        visto.add(x)
    return False


assert dos_suma([8, 2, 7, 5], 9) and not dos_suma([8, 2, 7, 5], 100)
assert not dos_suma([5], 10) and dos_suma([5, 5], 10)


# Índices de los dos números
def dos_suma_indices(arr, k):
    visto = {}
    for i, x in enumerate(arr):
        if k - x in visto:
            return visto[k - x], i
        visto[x] = i
    return None


assert dos_suma_indices([8, 2, 7, 5], 9) == (1, 2)

### Elemento más frecuente y su cantidad

In [ ]:
def mas_frecuente(arr):
    cuenta = Counter(arr)
    mejor = max(cuenta.values())
    return min(x for x, c in cuenta.items() if c == mejor), mejor        # el menor en caso de empate


assert mas_frecuente([3, 1, 3, 2, 1]) == (1, 2)

## ¿Qué herramienta uso?

| Si necesito... | Uso |
|---|---|
| ordenar o encontrar el mínimo/máximo/mediana | `sorted` |
| buscar en algo ordenado, o "el primer valor que cumple" | **búsqueda binaria** (`bisect`) |
| "el mayor/menor valor tal que funciona" | **búsqueda binaria sobre la respuesta** |
| elegir lo mejor en cada paso (ordenando primero) | **goloso** |
| emparejar/deshacer en orden inverso (paréntesis, "último abierto") | **pila** |
| procesar en orden de llegada, caminos más cortos sin pesos | **cola** (`deque`) y BFS |
| saber si algo ya apareció | **set** |
| contar o asociar datos | **dict** / `Counter` |

## Problemas tipo (estilo grader)

**P1.** `cuantas_veces(arr_ordenado, x)`: cuántas veces aparece `x` (con `bisect`).
**P2.** `tarea_con_mas_prioridad(...)`: dadas tareas `(prioridad, nombre)`, devuelve los nombres en orden de prioridad descendente y alfabético en empate.
**P3.** `rotacion_de_cola(n, k)`: simula `k` rotaciones de la lista `1..n` con `deque`.

In [ ]:
def cuantas_veces(arr, x):
    return bisect_right(arr, x) - bisect_left(arr, x)


def ordenar_tareas(tareas):
    return [n for p, n in sorted(tareas, key=lambda t: (-t[0], t[1]))]


def rotacion_de_cola(n, k):
    q = deque(range(1, n + 1))
    q.rotate(-k)
    return list(q)


assert cuantas_veces([1, 2, 2, 2, 5], 2) == 3 and cuantas_veces([1, 5], 3) == 0
assert ordenar_tareas([(2, "b"), (3, "z"), (2, "a")]) == ["z", "a", "b"]
assert rotacion_de_cola(5, 2) == [3, 4, 5, 1, 2]
print("ok")

## Para practicar

1. `raiz_entera(n)`: `⌊√n⌋` con búsqueda binaria (sin `**` ni `math`).
2. `intervalos_unidos(intervalos)`: unir intervalos que se traslapan (ordena por inicio).
3. `temperaturas_mas_calidas(t)`: para cada día, cuántos días hay que esperar hasta uno más cálido (pila).
4. `primer_caracter_repetido(s)`: con `set`.
5. `distancia_mas_corta(grid, inicio, fin)`: BFS con posiciones de inicio y fin dadas.
6. `monedas_minimas(monto, monedas)` con monedas arbitrarias: observa que el goloso falla (te prepara para programación dinámica).

## Errores típicos

- Búsqueda binaria sobre un arreglo **sin ordenar**.
- Bucle infinito por `medio = (izq + der) // 2` cuando `izq = medio`: usa `medio = (izq + der + 1) // 2` en ese caso.
- Aplicar un goloso sin demostrar que funciona (prueba con casos pequeños contra fuerza bruta).
- `list.pop(0)` en una cola grande.
- Modificar un `set`/`dict` mientras lo recorres.